## Backpropagation

Let's look at how Pytorch implements _backpropagation_. All we need to do is create some tensors, tell Pytorch that we want to compute gradients for them, and then do some operations on them that result in a single scalar.

Vamos analisar como o Pytorch implementa a _backpropagação_. Tudo o que precisamos fazer é criar alguns tensores, informar ao Pytorch que queremos calcular os gradientes para eles e, em seguida, realizar algumas operações que resultem em um único escalar.

In [ ]:
import torch

In [ ]:
a, b = torch.randn(3,4), torch.rand(3,4) # Criando alguns tensores (Rand = gera valores aleatórios entre 0 - 1,
# Randn distribuição gaussiana padrão)
a.requires_grad = True # ativando os gradientes 

out = ((a + b)**2).sum() # a função sum() retorna a soma de todos os elementos do iterável
print(out)


Pytorch has not just computed the result (```out```), it's also included a pointer to a ```SumBackward``` object representing the computation (summing) that created ```out```. This object links back to other objects, all the way down to the start of the computation.

Pytorch não apenas calculou o resultado (```out```), mas também incluiu um ponteiro para um objeto ```SumBackward``` que representa o cálculo (soma) que criou ```out```. Este objeto está vinculado a outros objetos, até chegar ao início do cálculo.

In [ ]:
print(out.grad_fn) # soma
print(out.grad_fn.next_functions[0][0]) # elevando a uma potencia
print(out.grad_fn.next_functions[0][0].next_functions[0][0])

# Observação: estes não são atributos que você normalmente usaria. Nós apenas os chamamos aqui para mostrar que
# o Pytorch está lembrando de tudo o que fazemos.

We've asked Pytorch to ensure we can compute a gradient on ```a``` and done some basic computation. The computation has resulted in a single number (```out```), so we can now compute the gradient of ```a``` over that output. Remember, backpropagation only works efficiently if the output of the computation graph is a single scalar, usually your loss.

Tradução: Pedimos ao Pytorch para garantir que possamos calcular um gradiente em ```a``` e fizemos alguns cálculos básicos. O cálculo resultou em um único número (```out```), então agora podemos calcular o gradiente de ```a``` em relação a essa saída. Lembre-se, a retropropagação só funciona eficientemente se a saída do grafo de cálculos for um único escalar, geralmente sua perda.

In [ ]:
print(a.grad) # this is gradient in a. Note it's currently empty (note que ele está vazio)

out.backward() # ask Pytorch to perform backpropagation (peça ao pytorch para realizar a backprop)

print(a.grad) # now a has a gradient(agr a possui um gradient)

Note that the gradient of ```a``` has the same shape as ```a```.

# Learning

Pytorch has many utilities to help you quickly build elaborate networks, but it's instructive to first see how you would use just these tools to build a simple model. As an example, we will build a simple linear regression model.

First, let's generate some simple random data.

PyTorch possui muitas utilidades para ajudá-lo a construir rapidamente redes elaboradas, mas é instrutivo ver como você usaria apenas essas ferramentas para construir um modelo simples. Como exemplo, vamos construir um modelo de regressão linear simples.

Primeiro, vamos gerar alguns dados aleatórios simples.

In [ ]:
x = torch.randn(1000,32) # 1000 instances, with 32 feactures 
wt, bt = torch.randn(32,1), torch.randn(1) # function to compute the true labels(etiquetas verdadeiras)
t = torch.mm(x,wt) + bt # the true target labels (entrada multiplicada por pesos e somada bias)

Next up, we define the parameters of our model (we'll initialize them randomly).

Em seguida, definimos os parametros do nosso modelo (vamos iniciá-los de forma aleatória)

In [ ]:
w = torch.randn(32,1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

Note that any method that creates tensors (like ```torch.randn()```) can be told that it should make them require a gradient).

Here's what one computation of the model output over the whole data looks like. We'll print the shapes of the tensors to see what's going on. **Before you run this cell, see if you can work out what the sizes will be.**

Observe que qualquer método que cria tensores (como ```torch.randn()```) pode ser informado de que eles devem exigir um gradiente.

Aqui está como uma computação da saída do modelo em todos os dados parece. Vamos imprimir as formas dos tensores para ver o que está acontecendo. **Antes de executar esta célula, tente descobrir quais serão os tamanhos.**

In [ ]:
print('data size', x.size())

# modelo de saída

y = torch.mm(x, w) + b 

print('output size:', y.size())

print()
print('the first 3 predictions', y[:3, 0])
print('Ground truth', t[:3,0]) # Ground truth é o termo usado para os dados reais
                                # note that these will be completely different, because
                                # we haven't started training yet
# residuals
r = t - y
print('residuals size', r.size())

# mean-square-error loss

loss = (r ** 2).mean() # Em Pytorch, a função mean() é usada para calcular a média de um tensor  
print()
print('loss:', loss.item())
# -- if you have a tensor with a single number, .item() will turn it into a normal float for you

We can now apply backpropagation, and see that we get a gradient over our two parameters ```w``` and ```b```. Before you run the cell, what will the sizes of the gradient tensors be?

Ao aplicarmos o backpropagation, podemos obter um gradiente em relação aos nossos dois parâmetros ```w``` e ```b```. Antes de executar a célula, qual será o tamanho dos tensores de gradiente?

In [ ]:
loss.backward() 

print('gradient on w', w.grad)
print('gradient on b', b.grad)

# NB: Se você executar a célula duas vezes, o PyTorch irá reclamar. Após cada operação de retropropagação (backward), o PyTorch espera uma nova operação de avanço (forward).

We are now ready to build a training loop. We'll use basic gradient descent without minibatches, computing the loss over the whole data every iteration. 

Agora estamos prontos para construir um loop de treinamento. Vamos usar o método básico de descida de gradiente sem mini lotes, calculando a perda em todo o conjunto de dados a cada iteração.

In [41]:
# Hiperparametros 
iterations = 21
learning_rate = 0.5

# Regenarate the data and model

x = torch.randn(1000, 32) # input
wt, bt = torch.randn(32,1), torch.randn(1) # parameters train
t = torch.mm(x, wt) + b # data train

# iniciando os parametros aleatórios e já ativando o gradiente 

w = torch.randn(32,1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

for i in  range(iterations):
    
    # Forward pass
    y = torch.mm(x,w) + b

    # mean-square-error
    r = t - y
    loss = (r ** 2).mean() 

    # backpropagation

    loss.backward()

    # print the loss

    print(f'iteration: {i:4}, loss: {loss:.4}')

    # Gradient descent

    w.data = w.data - learning_rate*w.grad.data
    b.data = b.data - learning_rate*b.grad.data

    # -- Observe que não queremos que o mecanismo de autodiferenciação calcule gradientes nesta parte.
    # Ao operar em w.data, estamos apenas alterando os valores do tensor, não
    # lembrando de um grafo de computação.

    # Detete the gradients
    w.grad.data.zero_()
    b.grad.data.zero_()

    # Se não fizermos isso, os gradientes são lembrados e quaisquer novos gradientes são adicionados aos antigos.

# show the true model, and the learned model 
print()
print('True model', wt.data[:4].t(), bt.data) # só transpôs com t() para melhor a vizu
print('Learned model', w.data[:4].t(), b.data)
    

iteration:    0, loss: 78.02
iteration:    1, loss: 3.1
iteration:    2, loss: 0.2035
iteration:    3, loss: 0.01596
iteration:    4, loss: 0.001375
iteration:    5, loss: 0.0001267
iteration:    6, loss: 1.236e-05
iteration:    7, loss: 1.269e-06
iteration:    8, loss: 1.363e-07
iteration:    9, loss: 1.521e-08
iteration:   10, loss: 1.753e-09
iteration:   11, loss: 2.075e-10
iteration:   12, loss: 2.486e-11
iteration:   13, loss: 3.347e-12
iteration:   14, loss: 5.791e-13
iteration:   15, loss: 1.805e-13
iteration:   16, loss: 5.832e-14
iteration:   17, loss: 2.113e-14
iteration:   18, loss: 7.176e-15
iteration:   19, loss: 2.345e-16
iteration:   20, loss: 0.0

True model tensor([[-0.7253],
        [ 1.0885],
        [ 0.3697],
        [-0.0117]]) tensor([-1.3317])
Learned model tensor([[-0.7253],
        [ 1.0885],
        [ 0.3697],
        [-0.0117]]) tensor([-0.7374])
